In [5]:
#!/usr/bin/env python
# coding: utf-8

import os, sys, time, sqlite3, smtplib
import pandas as pd
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formataddr

start_time = time.time()

try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR, GSHEET_NAME, SHEET_NAME  # noqa: E402

# ---- DB パス
user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == 'nt' else os.path.expanduser("~"),
    "myenv310", PROJECT_DIR
)
db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# ---- 表示・取得列
display_columns = [
    "台番号", "サイトURL", "pscubeURL", "ptoolURL", "ぱち解析URL", "期待値URL", "機種名",
    "大当り回数", "最終スタート", "宵越し累計ゲーム数"
]
need_columns = [
    "実行日", "台番号",
    "宵越し最終ステータス",
    "機種別回転設定値",
    "単発後天井残りゲーム数",
    "確変後天井残りゲーム数",
]
border_cols_all = [f"等価削りあり{n}回転プラマイボーダー残りゲーム数" for n in range(13, 21)]
threshold_cols = [
    "svg差枚最終回転率",
    "svg差枚noボナ回転率",
    "出玉数noボナ回転率",
    "出玉数初当り確率最終回転率",
]

# サイトURLはDBに無い列なので除外してSELECTを作る
db_columns = list(dict.fromkeys(
    [c for c in display_columns if c != "サイトURL"]
    + need_columns + border_cols_all + threshold_cols
))

# ---- 取得（最新日の台ごと最新行）
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(
        f'''SELECT {", ".join([f"[{c}]" for c in db_columns])}
            FROM result_table
            ORDER BY ROWID DESC''', conn
    )
if df.empty:
    print("❌ 対象データがありません"); sys.exit(0)

df["実行日"] = pd.to_datetime(df["実行日"], errors='coerce')
df = df.dropna(subset=["実行日"])
if df.empty:
    print("❌ 実行日が不正"); sys.exit(0)

max_date = df["実行日"].dt.date.max()
df_latest = (df[df["実行日"].dt.date == max_date]
             .sort_values("実行日", ascending=False)
             .drop_duplicates(subset=["台番号"])
             .copy())
print(f"[INFO] 最新日: {max_date} 件数: {len(df_latest)}")

# ---- サイトURL列（メール用）
base_url = f"https://sedoinfinity.xsrv.jp/{PROJECT_DIR}/machines/machine_"
df_latest["サイトURL"] = df_latest["台番号"].astype(str).map(
    lambda x: f'<a href="{base_url}{x}.html" target="_blank">リンク</a>'
)

# ---- メール
smtp_server, port = "smtp.gmail.com", 587
sender_email = "straydog12341234@gmail.com"
password = "cyam jmtc pgfr slpx"  # 本番は環境変数推奨
receiver_emails = ["straydog12341234@gmail.com", "selectshop@gmail.com"]

def send_email(subject, html_body):
    msg = MIMEMultipart("alternative")
    msg["From"] = formataddr((GSHEET_NAME, sender_email))
    msg["To"] = ", ".join(receiver_emails)
    msg["Subject"] = subject
    msg.attach(MIMEText(html_body, "html"))
    try:
        with smtplib.SMTP(smtp_server, port) as s:
            s.starttls(); s.login(sender_email, password)
            s.sendmail(sender_email, receiver_emails, msg.as_string())
        print(f"✅ メール送信: {subject}")
    except Exception as e:
        print(f"[ERROR] メール送信失敗: {e}")

# ---- URL列→アンカー（列名に'URL'を含む全列）
def convert_url_columns_to_anchor(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    url_cols = [c for c in df_out.columns if "URL" in str(c)]
    def to_anchor(v):
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("http://") or s.startswith("https://"):
                return f'<a href="{s}" target="_blank">リンク</a>'
        return v
    for col in url_cols:
        df_out[col] = df_out[col].apply(to_anchor)
    return df_out

# ---- 既存: 分岐チェック
def check_and_notify(df_base: pd.DataFrame, kind_label: str, remain_col: str):
    dfw = df_base.copy()
    if remain_col in dfw.columns:
        dfw[remain_col] = pd.to_numeric(dfw[remain_col], errors="coerce")
    if "機種別回転設定値" in dfw.columns:
        dfw["機種別回転設定値"] = pd.to_numeric(dfw["機種別回転設定値"], errors="coerce")
    if "宵越し最終ステータス" not in dfw.columns:
        return 0
    dfw = dfw[dfw["宵越し最終ステータス"] == kind_label].copy()

    rotation_to_col = {n: f"等価削りあり{n}回転プラマイボーダー残りゲーム数"
                       for n in range(13, 21)
                       if f"等価削りあり{n}回転プラマイボーダー残りゲーム数" in dfw.columns}
    if dfw.empty or not rotation_to_col:
        return 0

    total = 0
    for n, border_col in rotation_to_col.items():
        sub = dfw[dfw["機種別回転設定値"] == n].copy()
        sub[border_col] = pd.to_numeric(sub[border_col], errors="coerce")
        hit = sub[sub[remain_col].notna() & sub[border_col].notna() & (sub[remain_col] < sub[border_col])].copy()
        if hit.empty: continue
        cols = display_columns + ["機種別回転設定値", remain_col, border_col]
        cols = [c for c in dict.fromkeys(cols) if c in hit.columns]
        df_mail = convert_url_columns_to_anchor(hit.sort_values("台番号")[cols])
        html = f"""<html><head><style>
        .styled-table{{border-collapse:collapse;width:100%}}
        .styled-table th,.styled-table td{{border:1px solid #ccc;padding:5px;text-align:left}}
        .styled-table th{{background:#f2f2f2}}
        </style></head><body>
        <h2>{kind_label} 条件一致（設定{n}） {len(df_mail)}件 ({max_date})</h2>
        {df_mail.to_html(index=False, escape=False, border=1, classes="styled-table")}
        </body></html>"""
        send_email(f"{GSHEET_NAME}{SHEET_NAME} {kind_label} 設定{n} 条件一致 {max_date}", html)
        total += len(df_mail)
    return total

# ---- 追加: 回転率しきい値(>=19)
def notify_threshold_over(df_base: pd.DataFrame, threshold: float = 19.0) -> int:
    dfw = df_base.copy()
    exist_cols = [c for c in threshold_cols if c in dfw.columns]
    if not exist_cols:
        print("[INFO] 回転率列が取得できていません（スキップ）"); return 0
    for c in exist_cols:
        dfw[c] = pd.to_numeric(dfw[c], errors="coerce")
    hits = dfw[dfw[exist_cols].ge(threshold).any(axis=1)].copy()
    if hits.empty:
        print(f"[INFO] 回転率 {threshold} 以上なし"); return 0
    cols = [c for c in dict.fromkeys(display_columns + exist_cols) if c in hits.columns]
    df_mail = convert_url_columns_to_anchor(hits.sort_values("台番号")[cols])
    html = f"""<html><head><style>
    .styled-table{{border-collapse:collapse;width:100%}}
    .styled-table th,.styled-table td{{border:1px solid #ccc;padding:5px;text-align:left}}
    .styled-table th{{background:#f2f2f2}}
    </style></head><body>
    <h2>回転率しきい値アラート（≧{threshold}）{len(df_mail)}台 ({max_date})</h2>
    {df_mail.to_html(index=False, escape=False, border=1, classes="styled-table")}
    </body></html>"""
    send_email(f"{GSHEET_NAME}{SHEET_NAME} 回転率≧{threshold} アラート {max_date}（{len(df_mail)}台）", html)
    return len(df_mail)

# ---- 実行
hits_total = 0
hits_total += check_and_notify(df_latest, "初当り", "単発後天井残りゲーム数")
hits_total += check_and_notify(df_latest, "継続", "確変後天井残りゲーム数")
hits_total += notify_threshold_over(df_latest, threshold=19.0)
print(f"[INFO] 合計ヒット: {hits_total}")

print(f"[INFO] スクリプト完了（実行時間: {time.time() - start_time:.2f} 秒）")


[INFO] 使用DB: C:\Users\stray\myenv310\yanai-gaia-p\db\output.db
[INFO] 最新日: 2025-08-20 件数: 5
✅ メール送信: 柳井ガイアpachi 継続 設定17 条件一致 2025-08-20
✅ メール送信: 柳井ガイアpachi 回転率≧14.0 アラート 2025-08-20（2台）
[INFO] 合計ヒット: 3
[INFO] スクリプト完了（実行時間: 6.63 秒）
